# Minimum Silver Table 6: QASMBench Conditional Corrections
**Course:** TU Delft DSAIT4000 (Data Management & Engineering) — Assignment 1  
**Target Table:** `silver/qasmbench/conditional_correction.parquet`  
**Shared Provenance:** `results/part1/source_trace.parquet`


---

### Objectives & Contracts
1. **Schema Fidelity:** Build `silver/qasmbench/conditional_correction.parquet` where **one row represents one recovery operation controlled by a measured syndrome**.
2. **Strict PyArrow Types:**
    - `source_record_id`: `string` (stable link to the correction statement)
    - `circuit_id`: `string` (link to the Silver circuit)
    - `condition_register`: `string` (classical register used by the condition)
    - `condition_value`: `int64` (integer value that activates the operation)
    - `gate`: `string` (correction gate name)
    - `target_qubit`: `string` (qubit acted on by the correction)
3. **Shared Tracing Rule:** Append QASMBench conditional-correction records to `results/part1/source_trace.parquet` using `save_source_traces`.
4. **Dual-Lake Persistence:** Write Parquet locally and synchronize it to MinIO bucket `quantum-lake`.



**Background Definitions**

* **Condition Register:** Register where the syndrome or measurement result is stored.
* **Condition Value:** The exact value that the condition register needs to have before triggering a correction action.
* **Gate:** Specific quantum operation to apply to the target qubit.



In [1]:
import hashlib
import io
from pathlib import Path
import zipfile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.tracing import save_source_traces

BASE_DIR = Path("/workspace") if Path("/workspace").exists() else Path(".").resolve()
print(f"Base Directory: {BASE_DIR}")

settings = Settings.from_environment()
print(f"Lake Backend: {settings.lake_backend}")
print(f"MinIO Endpoint: {settings.s3_endpoint} (Bucket: {settings.s3_bucket})")

Base Directory: /workspace
Lake Backend: minio
MinIO Endpoint: http://minio:9000 (Bucket: quantum-lake)


### Step 1: Bronze ingestion and conditional-correction extraction

The curated archive contains source and transpiled variants. The parser records only explicit conditional recovery statements; it does not simulate the circuits or infer corrections that are not written in the QASM.

In [2]:
# Locate the QASMBench Bronze zip object (try MinIO first, fallback to local path)
bronze_object_name = "bronze/source=qasmbench/qasmbench-qec.zip"
bronze_bytes = None

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        print(f"Fetching '{bronze_object_name}' from MinIO...")
        response = client.get_object(settings.s3_bucket, bronze_object_name)
        bronze_bytes = response.read()
        response.close()
        response.release_conn()
        print(f"Successfully retrieved from MinIO ({len(bronze_bytes):,} bytes)")
    except Exception as e:
        print(f"MinIO fetch warning: {e}. Falling back to local file.")

if bronze_bytes is None:
    candidates = [
        Path("/course-data/raw/source=qasmbench/qasmbench-qec.zip"),
        BASE_DIR.parent / "datasets/student-bundle/core/raw/source=qasmbench/qasmbench-qec.zip",
        Path("datasets/student-bundle/core/raw/source=qasmbench/qasmbench-qec.zip"),
    ]
    for path in candidates:
        if path.exists():
            print(f"Reading from local path: {path}")
            bronze_bytes = path.read_bytes()
            break

assert bronze_bytes is not None, "Could not locate the QASMBench archive!"
bronze_sha256 = hashlib.sha256(bronze_bytes).hexdigest()
print(f"QASMBench Bronze Archive SHA-256: {bronze_sha256}")
print(f"QASMBench Bronze Archive Size:    {len(bronze_bytes):,} bytes")

Fetching 'bronze/source=qasmbench/qasmbench-qec.zip' from MinIO...
Successfully retrieved from MinIO (144,172 bytes)
QASMBench Bronze Archive SHA-256: 60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e4f85722a
QASMBench Bronze Archive Size:    144,172 bytes


### Step 2: Parse corrections and record stable lineage

Each correction row points to the exact archive member and statement position that defines it.

In [3]:
import re

CONDITIONAL_CORRECTION = re.compile(
    r"^if\s*\(\s*([A-Za-z_]\w*)\s*==\s*(-?\d+)\s*\)\s*([A-Za-z_]\w*)\s+([^;]+)$",
    re.IGNORECASE,
)


def parse_conditional_corrections(member, text):
    cleaned = re.sub(r"//.*", "", text)
    circuit_id = f"qasmbench:{member[:-5]}"
    records = []
    for statement_index, statement in enumerate(cleaned.split(";")):
        statement = " ".join(statement.split())
        if not statement:
            continue
        match = CONDITIONAL_CORRECTION.fullmatch(statement)
        if not match:
            continue
        condition_register, condition_value, gate, target_qubit = match.groups()
        records.append({
            "source_record_id": f"qasmbench:{member}:statement:{statement_index}",
            "circuit_id": circuit_id,
            "condition_register": condition_register,
            "condition_value": int(condition_value),
            "gate": gate,
            "target_qubit": "".join(target_qubit.split()),
            "archive_member": member,
            "record_locator": f"statement:{statement_index}",
        })
    return records


correction_records = []
trace_records = []
with zipfile.ZipFile(io.BytesIO(bronze_bytes)) as archive:
    qasm_members = sorted(name for name in archive.namelist() if name.endswith(".qasm"))
    for member in qasm_members:
        records = parse_conditional_corrections(member, archive.read(member).decode("utf-8"))
        correction_records.extend(records)
        for record in records:
            trace_records.append({
                "source_record_id": record["source_record_id"],
                "source_name": "qasmbench",
                "bronze_object": bronze_object_name,
                "archive_member": record["archive_member"],
                "record_locator": record["record_locator"],
                "input_sha256": bronze_sha256,
            })
        print(f"  {Path(member).parent.name}/{Path(member).stem}: {len(records)} conditional corrections")

assert len(correction_records) == 6
assert len({record["source_record_id"] for record in correction_records}) == len(correction_records)
print(f"\nExtracted {len(correction_records)} conditional corrections")

  error_correctiond3_n5/error_correctiond3_n5: 0 conditional corrections
  error_correctiond3_n5/error_correctiond3_n5_transpiled: 0 conditional corrections
  qec_en_n5/qec_en_n5: 0 conditional corrections
  qec_en_n5/qec_en_n5_transpiled: 0 conditional corrections
  qec_sm_n5/qec_sm_n5: 3 conditional corrections
  qec_sm_n5/qec_sm_n5_transpiled: 3 conditional corrections

Extracted 6 conditional corrections


### Step 3: Strict Arrow schema and Parquet export

The condition value is stored as `int64`, while the register, gate, and target remain strings so the original QASM meaning is preserved.

In [4]:
conditional_correction_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("circuit_id", pa.string()),
    ("condition_register", pa.string()),
    ("condition_value", pa.int64()),
    ("gate", pa.string()),
    ("target_qubit", pa.string()),
])

table_corrections = pa.Table.from_pandas(
    pd.DataFrame(correction_records),
    schema=conditional_correction_schema,
    preserve_index=False,
)
assert table_corrections.schema.equals(conditional_correction_schema)
print("=== QASMBench Conditional Correction Table ===")
print(f"Rows: {table_corrections.num_rows}, Columns: {table_corrections.num_columns}")
print(table_corrections.schema)

silver_dir = BASE_DIR / "silver/qasmbench"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_parquet_path = silver_dir / "conditional_correction.parquet"

results_dir = BASE_DIR / "results/part1"
results_dir.mkdir(parents=True, exist_ok=True)
trace_parquet_path = results_dir / "source_trace.parquet"

pq.write_table(table_corrections, silver_parquet_path, compression="zstd")
print(f"Wrote Silver conditional-correction table to: {silver_parquet_path} ({silver_parquet_path.stat().st_size:,} bytes)")

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        minio_key = "silver/qasmbench/conditional_correction.parquet"
        client.fput_object(settings.s3_bucket, minio_key, str(silver_parquet_path))
        print(f"Uploaded Silver table to MinIO: {settings.s3_bucket}/{minio_key}")
    except Exception as e:
        print(f"Warning: MinIO upload failed: {e}")

trace_count = save_source_traces(
    new_records=trace_records,
    source_name="qasmbench",
    trace_file_path=trace_parquet_path,
    settings=settings,
)
print(f"Master source_trace table updated; total rows: {trace_count:,}")

saved = pq.read_table(silver_parquet_path)
assert saved.schema.equals(conditional_correction_schema)
assert saved.num_rows == len(correction_records)
print("Table 6 validation passed.")

=== QASMBench Conditional Correction Table ===
Rows: 6, Columns: 6
source_record_id: string
circuit_id: string
condition_register: string
condition_value: int64
gate: string
target_qubit: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 822
Wrote Silver conditional-correction table to: /workspace/silver/qasmbench/conditional_correction.parquet (4,514 bytes)
Uploaded Silver table to MinIO: quantum-lake/silver/qasmbench/conditional_correction.parquet
Master source_trace table updated; total rows: 325,623
Table 6 validation passed.


In [5]:
import pyarrow.parquet as pq

circuit_path = "/workspace/silver/qasmbench/conditional_correction.parquet"
trace_path = "/workspace/results/part1/source_trace.parquet"

circuits = pq.read_table(circuit_path)
traces = pq.read_table(trace_path)

print("Stabilizer checks table")
print("Rows:", circuits.num_rows)
print("Columns:", circuits.column_names)
print(circuits.schema)
print(circuits.to_pandas().to_string(index=False))

print("\nQASMBench traces")
trace_df = traces.to_pandas()
qasmbench_traces = trace_df[trace_df["source_name"] == "qasmbench"]
print("Rows:", len(qasmbench_traces))
print(qasmbench_traces.to_string(index=False))

Stabilizer checks table
Rows: 6
Columns: ['source_record_id', 'circuit_id', 'condition_register', 'condition_value', 'gate', 'target_qubit']
source_record_id: string
circuit_id: string
condition_register: string
condition_value: int64
gate: string
target_qubit: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 822
                                                source_record_id                                     circuit_id condition_register  condition_value gate target_qubit
           qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:statement:14            qasmbench:small/qec_sm_n5/qec_sm_n5                syn                1    x         q[0]
           qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:statement:15            qasmbench:small/qec_sm_n5/qec_sm_n5                syn                2    x         q[2]
           qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:statement:16            qasmbench:small/qec_sm_n5/qec_sm_n5                syn